# Выполнила студентка группы 6401 Иванова Дарья

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from math import radians, sin, cos, sqrt, atan2

# Spark
spark = SparkSession.builder \
    .appName("bike_tasks") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Чтение данных
trips = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y H:m") \
    .csv("file:///root/trip.csv")

stations = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("timestampFormat", "M/d/y") \
    .csv("file:///root/station.csv")

In [2]:
# -----------------------------
# 1. Велосипед с максимальным временем пробега
# -----------------------------
bike_max_time = (
    trips.groupBy("bike_id")
    .agg(F.sum("duration").alias("total_duration_sec"))
    .orderBy(F.desc("total_duration_sec"))
)

print("1) Велосипед с максимальным временем пробега:")
bike_max_time.show(1, truncate=False)

max_bike_id = bike_max_time.first()["bike_id"]

1) Велосипед с максимальным временем пробега:


+-------+------------------+
|bike_id|total_duration_sec|
+-------+------------------+
|535    |18611693          |
+-------+------------------+
only showing top 1 row



In [3]:
# -----------------------------
# 2. Наибольшее геодезическое расстояние между станциями
# -----------------------------
# Haversine в км
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return float(R * c)

haversine_udf = F.udf(haversine, "double")

s1 = stations.select(
    F.col("id").alias("id1"),
    F.col("name").alias("name1"),
    F.col("lat").alias("lat1"),
    F.col("long").alias("lon1")
)

s2 = stations.select(
    F.col("id").alias("id2"),
    F.col("name").alias("name2"),
    F.col("lat").alias("lat2"),
    F.col("long").alias("lon2")
)

max_station_distance = (
    s1.crossJoin(s2)
    .where(F.col("id1") < F.col("id2"))
    .withColumn("distance_km", haversine_udf("lat1", "lon1", "lat2", "lon2"))
    .orderBy(F.desc("distance_km"))
)

print("2) Наибольшее геодезическое расстояние между станциями:")
max_station_distance.select("distance_km").show(1, truncate=False)
max_station_distance.select("id1", "name1", "id2", "name2", "distance_km").show(1, truncate=False)

2) Наибольшее геодезическое расстояние между станциями:


+----------------+
|distance_km     |
+----------------+
|69.9208759542826|
+----------------+
only showing top 1 row

+---+--------------------------+---+----------------------+----------------+
|id1|name1                     |id2|name2                 |distance_km     |
+---+--------------------------+---+----------------------+----------------+
|16 |SJSU - San Salvador at 9th|60 |Embarcadero at Sansome|69.9208759542826|
+---+--------------------------+---+----------------------+----------------+
only showing top 1 row



In [4]:
# -----------------------------
# 3. Путь велосипеда с максимальным временем пробега через станции
# -----------------------------
bike_path = (
    trips.filter(F.col("bike_id") == max_bike_id)
    .orderBy("start_date")
    .select(
        "bike_id",
        "start_date",
        "end_date",
        "start_station_name",
        "end_station_name",
        "duration"
    )
)

print("3) Путь велосипеда с максимальным временем пробега через станции:")
bike_path.show(200, truncate=False)

# Путь в виде строки:
bike_edges = bike_path.select(
    F.concat_ws(" -> ", F.col("start_station_name"), F.col("end_station_name")).alias("segment")
)
segments = [row["segment"] for row in bike_edges.collect()]
print("Маршрут:")
for seg in segments:
    print(seg)

3) Путь велосипеда с максимальным временем пробега через станции:


+-------+-------------------+-------------------+---------------------------------------------+---------------------------------------------+--------+
|bike_id|start_date         |end_date           |start_station_name                           |end_station_name                             |duration|
+-------+-------------------+-------------------+---------------------------------------------+---------------------------------------------+--------+
|535    |2013-08-29 19:32:00|2013-08-29 19:53:00|Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)     |1245    |
|535    |2013-08-29 21:38:00|2013-08-29 21:45:00|San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend)      |423     |
|535    |2013-08-30 08:40:00|2013-08-30 08:54:00|San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                            |842     |
|535    |2013-08-30 09:10:00|2013-08-30 09:19:00|Market at Sansome                            

[Stage 16:>                                                       (0 + 10) / 10]

Маршрут:
Post at Kearney -> San Francisco Caltrain (Townsend at 4th)
San Francisco Caltrain (Townsend at 4th) -> San Francisco Caltrain 2 (330 Townsend)
San Francisco Caltrain 2 (330 Townsend) -> Market at Sansome
Market at Sansome -> 2nd at South Park
2nd at Townsend -> Davis at Jackson
San Francisco City Hall -> Civic Center BART (7th at Market)
Civic Center BART (7th at Market) -> Post at Kearney
Post at Kearney -> Embarcadero at Sansome
Embarcadero at Sansome -> Washington at Kearney
Washington at Kearney -> Market at Sansome
Market at Sansome -> Market at Sansome
Market at Sansome -> 2nd at Folsom
2nd at Folsom -> 2nd at Townsend
Temporary Transbay Terminal (Howard at Beale) -> 2nd at Townsend
2nd at Townsend -> Embarcadero at Sansome
Embarcadero at Sansome -> Clay at Battery
Clay at Battery -> Harry Bridges Plaza (Ferry Building)
Harry Bridges Plaza (Ferry Building) -> Clay at Battery
Clay at Battery -> San Francisco Caltrain (Townsend at 4th)
San Francisco Caltrain (Townsend at 

In [5]:
# -----------------------------
# 4. Количество велосипедов в системе
# -----------------------------
bike_count = trips.select("bike_id").distinct().count()
print("4) Количество велосипедов в системе:", bike_count)

[Stage 19:>                                                       (0 + 10) / 10]

4) Количество велосипедов в системе: 700


In [6]:
# -----------------------------
# 5. Пользователи, потратившие на поездки более 3 часов
# -----------------------------
# В датасете нет user_id, используем zip_code как прокси
users_over_3h = (
    trips.filter(F.col("zip_code").isNotNull() & (F.col("zip_code") != ""))
    .groupBy("zip_code")
    .agg(F.sum("duration").alias("total_duration_sec"))
    .filter(F.col("total_duration_sec") > 3 * 3600)
    .orderBy(F.desc("total_duration_sec"))
)

print("5) Пользователи (по zip_code), потратившие более 3 часов:")
users_over_3h.show(users_over_3h.count(), truncate=False)

spark.stop()

5) Пользователи (по zip_code), потратившие более 3 часов:


+----------+------------------+
|zip_code  |total_duration_sec|
+----------+------------------+
|94107     |49757162          |
|nil       |45725550          |
|94105     |25596128          |
|94133     |21637675          |
|94102     |19128021          |
|94103     |19127388          |
|95531     |17270400          |
|94111     |14244997          |
|95112     |12742370          |
|94109     |12057128          |
|94040     |7807926           |
|94110     |7421936           |
|94117     |6901313           |
|94301     |6590378           |
|94041     |6276284           |
|94158     |6248167           |
|94306     |5550643           |
|94025     |5178237           |
|94108     |5127562           |
|94611     |5014906           |
|94010     |4800158           |
|94403     |4686613           |
|95110     |4573785           |
|94114     |4285860           |
|94501     |4086593           |
|94602     |3874468           |
|94303     |3860059           |
|95126     |3692719           |
|94112  